In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
import joblib
import json
import os

# Загрузка данных
df = pd.read_csv('S06-hw-dataset-01.csv', index_col=0)

# Первичный анализ
print(df.head())
print(df.info())

       num01     num02     num03     num04     num05     num06     num07  \
id                                                                         
1  -0.946058 -0.070313  1.824445 -2.754422  0.808865 -0.111094 -0.268950   
2  -2.484027  0.739378  1.596908 -2.586479 -0.033225 -3.054412 -4.706908   
3   1.522629  7.159635 -0.564903 -4.493110  1.623610  5.450187 -0.974595   
4   0.463373 -1.073908  1.752813  0.362786  2.790872  4.082385  0.322283   
5   3.188390 -4.701692 -0.689918 -0.448995  0.373821 -3.275363 -1.760931   

       num08     num09     num10  ...     num20     num21     num22     num23  \
id                                ...                                           
1  -3.078210  0.801275 -0.417059  ... -1.616515 -1.989464  1.407390 -0.218362   
2  -9.795169  0.145911 -1.122641  ... -1.727040 -0.583997  1.136761  0.285978   
3  -5.189589  1.600591 -0.477243  ...  0.524408  2.022430  1.278358 -0.850547   
4   3.390984 -0.033929  1.149337  ...  2.399834 -1.431576 -0.7

In [2]:
print(df.describe(include='all'))

              num01         num02         num03         num04         num05  \
count  12000.000000  12000.000000  12000.000000  12000.000000  12000.000000   
mean       0.013705     -0.005278     -0.002357      0.038404     -0.006301   
std        2.096534      3.544498      1.004417      2.087318      1.007329   
min       -8.155181    -14.605579     -4.374043     -7.753301     -3.999332   
25%       -1.423844     -2.303772     -0.685618     -1.374579     -0.676552   
50%       -0.047565      0.053348     -0.009186      0.038766     -0.003728   
75%        1.422676      2.335937      0.671441      1.484417      0.669269   
max        8.610863     16.299709      3.651692      7.571965      4.562115   

              num06         num07         num08         num09         num10  \
count  12000.000000  12000.000000  12000.000000  12000.000000  12000.000000   
mean      -0.865297     -0.702877     -0.290694     -0.008154      0.010048   
std        3.888966      1.989513      3.455981    

In [3]:
# Распределение таргета
target_col = 'target'
print(df[target_col].value_counts(normalize=True))

target
0    0.676583
1    0.323417
Name: proportion, dtype: float64


In [4]:
# Пропуски и типы
print(df.isnull().sum())
print(df.dtypes)

num01            0
num02            0
num03            0
num04            0
num05            0
num06            0
num07            0
num08            0
num09            0
num10            0
num11            0
num12            0
num13            0
num14            0
num15            0
num16            0
num17            0
num18            0
num19            0
num20            0
num21            0
num22            0
num23            0
num24            0
cat_contract     0
cat_region       0
cat_payment      0
tenure_months    0
target           0
dtype: int64
num01            float64
num02            float64
num03            float64
num04            float64
num05            float64
num06            float64
num07            float64
num08            float64
num09            float64
num10            float64
num11            float64
num12            float64
num13            float64
num14            float64
num15            float64
num16            float64
num17            float64
num18      

In [5]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [7]:
# Baseline 1: Dummy (most_frequent)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

# Baseline 2: Logistic Regression (с масштабированием)
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42, max_iter=1000))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None

In [8]:
def compute_metrics(y_true, y_pred, y_proba=None, average='binary'):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average=average)
    }
    if y_proba is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_proba, multi_class='ovr')
        except:
            metrics['roc_auc'] = None
    return metrics

In [9]:
dt_params = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 2, 5, 10]
}
dt = DecisionTreeClassifier(random_state=42)
dt_cv = GridSearchCV(dt, dt_params, cv=5, scoring='f1_macro' if len(np.unique(y)) > 2 else 'f1')
dt_cv.fit(X_train, y_train)
dt_best = dt_cv.best_estimator_
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 2, 5]
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_cv = GridSearchCV(rf, rf_params, cv=5, scoring='f1_macro' if len(np.unique(y)) > 2 else 'f1')
rf_cv.fit(X_train, y_train)
rf_best = rf_cv.best_estimator_
gb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}
gb = GradientBoostingClassifier(random_state=42)
gb_cv = GridSearchCV(gb, gb_params, cv=5, scoring='f1_macro' if len(np.unique(y)) > 2 else 'f1')
gb_cv.fit(X_train, y_train)
gb_best = gb_cv.best_estimator_
estimators = [
    ('dt', dt_best),
    ('rf', rf_best),
    ('gb', gb_best)
]
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42, max_iter=1000),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('dt',
                                DecisionTreeClassifier(max_depth=10,
                                                       min_samples_leaf=5,
                                                       random_state=42)),
                               ('rf',
                                RandomForestClassifier(n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(max_depth=5,
                                                           n_estimators=200,
                                                           random_state=42))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      random_state=42),
                   n_jobs=-1)

In [10]:
models = {
    'Dummy': (dummy, None),
    'LogisticRegression': (lr_pipe, lr_pipe.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None),
    'DecisionTree': (dt_best, dt_best.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None),
    'RandomForest': (rf_best, rf_best.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None),
    'GradientBoosting': (gb_best, gb_best.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None),
}

if 'stacking' in locals():
    models['Stacking'] = (
        stacking,
        stacking.predict_proba(X_test)[:, 1] if len(np.unique(y)) == 2 else None
    )

# Определение average для F1
average = 'binary' if len(np.unique(y)) == 2 else 'macro'

metrics_all = {}
for name, (model, y_proba) in models.items():
    y_pred = model.predict(X_test)
    metrics = compute_metrics(y_test, y_pred, y_proba, average=average)
    metrics_all[name] = metrics
    print(f"{name}")
    print(metrics)
    print(classification_report(y_test, y_pred))

Dummy
{'accuracy': 0.6766666666666666, 'f1': 0.0}
              precision    recall  f1-score   support

           0       0.68      1.00      0.81      1624
           1       0.00      0.00      0.00       776

    accuracy                           0.68      2400
   macro avg       0.34      0.50      0.40      2400
weighted avg       0.46      0.68      0.55      2400

LogisticRegression
{'accuracy': 0.8275, 'f1': 0.7076271186440678, 'roc_auc': np.float64(0.8746905312071505)}
              precision    recall  f1-score   support

           0       0.84      0.91      0.88      1624
           1       0.78      0.65      0.71       776

    accuracy                           0.83      2400
   macro avg       0.81      0.78      0.79      2400
weighted avg       0.82      0.83      0.82      2400

DecisionTree
{'accuracy': 0.8720833333333333, 'f1': 0.7924273157538878, 'roc_auc': np.float64(0.8950488167182977)}
              precision    recall  f1-score   support

           0     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


RandomForest
{'accuracy': 0.9291666666666667, 'f1': 0.8854447439353099, 'roc_auc': np.float64(0.9672947031638819)}
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1624
           1       0.93      0.85      0.89       776

    accuracy                           0.93      2400
   macro avg       0.93      0.91      0.92      2400
weighted avg       0.93      0.93      0.93      2400

GradientBoosting
{'accuracy': 0.9291666666666667, 'f1': 0.8868175765645806, 'roc_auc': np.float64(0.9689174305520287)}
              precision    recall  f1-score   support

           0       0.93      0.96      0.95      1624
           1       0.92      0.86      0.89       776

    accuracy                           0.93      2400
   macro avg       0.93      0.91      0.92      2400
weighted avg       0.93      0.93      0.93      2400

Stacking
{'accuracy': 0.93375, 'f1': 0.8956007879185818, 'roc_auc': np.float64(0.9695316070793764)}
            

In [11]:
# Confusion Matrix (лучшая модель по F1 или ROC-AUC)
best_model_name = max(metrics_all, key=lambda k: metrics_all[k].get('roc_auc', metrics_all[k]['f1']))
best_model, _ = models[best_model_name]

# Confusion Matrix
cm = confusion_matrix(y_test, best_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f'Confusion Matrix: {best_model_name}')
plt.savefig('confusion_matrix.png')
plt.close()

# ROC Curve (только для бинарной классификации)
if len(np.unique(y)) == 2:
    y_proba_best = best_model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_proba_best)
    plt.title(f'ROC Curve: {best_model_name}')
    plt.savefig('roc_curve.png')
    plt.close()

    # PR Curve (по желанию, особенно при дисбалансе)
    PrecisionRecallDisplay.from_predictions(y_test, y_proba_best)
    plt.title(f'PR Curve: {best_model_name}')
    plt.savefig('pr_curve.png')
    plt.close()

In [ ]:
# Permutation Importance для лучшей модели
perm_imp = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

# Топ-15 признаков
indices = np.argsort(perm_imp.importances_mean)[::-1][:15]
feature_names = X.columns[indices]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), perm_imp.importances_mean[indices])
plt.yticks(range(len(indices)), feature_names)
plt.gca().invert_yaxis()
plt.title('Permutation Importance (Top 15)')
plt.tight_layout()
plt.savefig('permutation_importance.png')
plt.close()

# Сохраняем важности
importance_dict = dict(zip(feature_names, perm_imp.importances_mean[indices]))

In [ ]:
artifacts_dir = 'artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

# 1. metrics_test.json
with open(f'{artifacts_dir}/metrics_test.json', 'w') as f:
    json.dump(metrics_all, f, indent=4)

# 2. search_summaries.json
search_summaries = {}
if 'dt_cv' in locals():
    search_summaries['DecisionTree'] = {
        'best_params': dt_cv.best_params_,
        'best_score': float(dt_cv.best_score_)
    }
if 'rf_cv' in locals():
    search_summaries['RandomForest'] = {
        'best_params': rf_cv.best_params_,
        'best_score': float(rf_cv.best_score_)
    }
if 'gb_cv' in locals():
    search_summaries['GradientBoosting'] = {
        'best_params': gb_cv.best_params_,
        'best_score': float(gb_cv.best_score_)
    }

with open(f'{artifacts_dir}/search_summaries.json', 'w') as f:
    json.dump(search_summaries, f, indent=4)

# 3. best_model.joblib
joblib.dump(best_model, f'{artifacts_dir}/best_model.joblib')

# 4. best_model_meta.json
best_meta = {
    'model_name': best_model_name,
    'model_params': best_model.get_params(),
    'test_metrics': metrics_all[best_model_name],
    'feature_importance_top15': importance_dict
}
with open(f'{artifacts_dir}/best_model_meta.json', 'w') as f:
    json.dump(best_meta, f, indent=4, default=str)